# 05 — Compute KPIs & Final Data Load

> **Notebook purpose:**  
> Ingest all cleaned hospital parquet/CSV files, derive a rich set of business KPIs/flags, run data-quality
> assertions, build cross-dimensional aggregation tables, and emit every artifact the Visualization Lead
> needs to build Tableau dashboards — all reproducibly, with zero magic numbers.
>
> **Outputs written to `data/processed/`**  
> | File | Description |
> |---|---|
> | `hospital_tableau_ready.csv` | Full fact table with all KPI flags & indices |
> | `analysis/kpi_scorecard.csv` | One-row-per-dataset executive scorecard |
> | `analysis/tableau_hospital_summary.csv` | Hospital-level KPI rollup |
> | `analysis/tableau_diagnosis_summary.csv` | Diagnosis-level KPI rollup |
> | `analysis/tableau_region_summary.csv` | Region-level KPI rollup |
> | `analysis/tableau_age_group_summary.csv` | Age-group KPI rollup |
> | `analysis/tableau_economic_status_summary.csv` | Economic-status KPI rollup |
> | `analysis/tableau_insurance_summary.csv` | Insurance-type KPI rollup |
> | `analysis/tableau_cross_dim_summary.csv` | Multi-dim cross-tab (diagnosis × age × insurance) |
> | `analysis/tableau_cost_tier_summary.csv` | Cost-tier KPI rollup |
> | `analysis/schema_manifest.csv` | Column-by-column data dictionary |


## 0 — Imports & Path Setup

In [1]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ── Paths ────────────────────────────────────────────────────────────────────
PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ANALYSIS_DIR  = PROCESSED_DIR / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Processed dir: {PROCESSED_DIR}')
print(f'Analysis dir : {ANALYSIS_DIR}')

Project root : /Users/harshita/SectionB_G11_HealthOps_Analytics
Processed dir: /Users/harshita/SectionB_G11_HealthOps_Analytics/data/processed
Analysis dir : /Users/harshita/SectionB_G11_HealthOps_Analytics/data/processed/analysis


/Users/harshita/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/harshita/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1 — Load & Merge All Cleaned Files

In [2]:
cleaned_paths = sorted(PROCESSED_DIR.glob('cleaned_hospital_*.csv'))
assert cleaned_paths, 'No cleaned_hospital_*.csv files found in PROCESSED_DIR'

frames = []
for p in cleaned_paths:
    chunk = pd.read_csv(p)
    chunk['_source_file'] = p.name          # lineage tag
    frames.append(chunk)
    print(f'  Loaded {p.name:40s} → {chunk.shape}')

df = pd.concat(frames, ignore_index=True)
print(f'\n✅ Combined shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

  Loaded cleaned_hospital_1.csv                   → (8000, 18)
  Loaded cleaned_hospital_2.csv                   → (8000, 18)
  Loaded cleaned_hospital_3.csv                   → (8000, 18)

✅ Combined shape: 24,000 rows × 18 columns


## 2 — Data-Quality Assertions
Hard-fail fast on critical integrity issues before any transformation.

In [3]:
REQUIRED_COLS = [
    'Patient_ID', 'Age', 'Gender', 'Hospital', 'Hospital_Type',
    'Diagnosis', 'Treatment_Cost', 'Doctor_Experience_Years',
    'Doctor_Availability', 'Cleanliness_Score', 'Economic_Status',
    'Outcome', 'Insurance', 'Region', 'Hospital_ID', 'Age_Group', 'Cost_Tier'
]

missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing_cols, f'Missing required columns: {missing_cols}'
print('✅ All required columns present.')

# Duplicate Patient IDs (warn only — same patient may appear across files)
dup_pct = df['Patient_ID'].duplicated().mean() * 100
print(f'   Duplicate Patient_ID: {dup_pct:.2f}%')

# Null-check on critical columns
critical_nulls = df[REQUIRED_COLS].isnull().sum()
critical_nulls = critical_nulls[critical_nulls > 0]
if len(critical_nulls):
    print(f'\n⚠️  Nulls in critical columns:\n{critical_nulls}')
else:
    print('✅ No nulls in critical columns.')

# Domain checks
valid_outcomes   = {'Recovered', 'Deceased', 'Under Treatment'}
valid_hosp_types = {'Private', 'Government'}
valid_insurance  = {'Yes', 'No'}

bad_outcome   = ~df['Outcome'].isin(valid_outcomes)
bad_hosp_type = ~df['Hospital_Type'].isin(valid_hosp_types)
bad_insurance = ~df['Insurance'].isin(valid_insurance)

for label, mask in [('Outcome', bad_outcome),
                     ('Hospital_Type', bad_hosp_type),
                     ('Insurance', bad_insurance)]:
    n = mask.sum()
    icon = '✅' if n == 0 else '⚠️ '
    print(f'{icon} {label}: {n} unexpected values')

assert df['Treatment_Cost'].ge(0).all(), 'Negative Treatment_Cost detected!'
print('✅ Treatment_Cost ≥ 0 for all rows.')

print(f'\n📊 Final shape after QC: {df.shape}')

✅ All required columns present.
   Duplicate Patient_ID: 66.67%
✅ No nulls in critical columns.
✅ Outcome: 0 unexpected values
✅ Hospital_Type: 0 unexpected values
✅ Insurance: 0 unexpected values
✅ Treatment_Cost ≥ 0 for all rows.

📊 Final shape after QC: (24000, 18)


## 3 — Outcome & Demographic Flags

In [4]:
tdf = df.copy()

# ── Outcome flags
tdf['Recovered_Flag']       = (tdf['Outcome'] == 'Recovered').astype(np.int8)
tdf['Deceased_Flag']        = (tdf['Outcome'] == 'Deceased').astype(np.int8)
tdf['Under_Treatment_Flag'] = (tdf['Outcome'] == 'Under Treatment').astype(np.int8)

# ── Payer / facility flags
tdf['Insured_Flag']          = (tdf['Insurance'] == 'Yes').astype(np.int8)
tdf['Private_Hospital_Flag'] = (tdf['Hospital_Type'] == 'Private').astype(np.int8)

# ── Gender flag (for quick pivots in Tableau)
tdf['Male_Flag']    = (tdf['Gender'] == 'Male').astype(np.int8)
tdf['Female_Flag']  = (tdf['Gender'] == 'Female').astype(np.int8)

# ── Vulnerability proxy (Senior + Low income + Uninsured)
tdf['Vulnerable_Patient_Flag'] = (
    (tdf['Age_Group'] == 'Senior') &
    (tdf['Economic_Status'] == 'Low') &
    (tdf['Insurance'] == 'No')
).astype(np.int8)

print('Outcome distribution:')
print(tdf['Outcome'].value_counts(normalize=True).mul(100).round(2).to_string())

Outcome distribution:
Outcome
Recovered         82.56
Under Treatment   12.56
Deceased           4.88


## 4 — Cost KPIs

In [5]:
# ── Variance vs. global mean
overall_avg_cost   = tdf['Treatment_Cost'].mean()
overall_median_cost = tdf['Treatment_Cost'].median()

tdf['Cost_vs_Overall_Avg']    = (tdf['Treatment_Cost'] - overall_avg_cost).round(2)
tdf['Cost_Pct_vs_Overall_Avg'] = ((tdf['Treatment_Cost'] / overall_avg_cost - 1) * 100).round(2)

# ── Variance vs. diagnosis-group mean
diag_avg          = tdf.groupby('Diagnosis')['Treatment_Cost'].transform('mean')
tdf['Cost_vs_Diagnosis_Avg']  = (tdf['Treatment_Cost'] - diag_avg).round(2)

# ── Variance vs. hospital-group mean
hosp_avg           = tdf.groupby('Hospital')['Treatment_Cost'].transform('mean')
tdf['Cost_vs_Hospital_Avg']   = (tdf['Treatment_Cost'] - hosp_avg).round(2)

# ── Cost percentile rank (global, 0-100)
tdf['Cost_Percentile_Rank']   = tdf['Treatment_Cost'].rank(pct=True).mul(100).round(1)

# ── High-cost flag: top 10% globally
cost_90th = tdf['Treatment_Cost'].quantile(0.90)
tdf['High_Cost_Flag']         = (tdf['Treatment_Cost'] >= cost_90th).astype(np.int8)

print(f'Overall avg cost  : ₹{overall_avg_cost:,.2f}')
print(f'Overall median cost: ₹{overall_median_cost:,.2f}')
print(f'90th-pct threshold : ₹{cost_90th:,.2f}')

Overall avg cost  : ₹24,849.47
Overall median cost: ₹23,413.86
90th-pct threshold : ₹41,329.23


## 5 — Operational Quality Indices

In [6]:
# ── Operational Access Index  (weighted composite, 0–1)
# Weights: doctor availability 50%, cleanliness 30%, experience 20%
tdf['Operational_Access_Index'] = (
    tdf['Doctor_Availability'].rank(pct=True) * 0.50
  + tdf['Cleanliness_Score'].rank(pct=True)   * 0.30
  + tdf['Doctor_Experience_Years'].rank(pct=True) * 0.20
).round(4)

# ── Quality tier based on composite index
tdf['Quality_Tier'] = pd.cut(
    tdf['Operational_Access_Index'],
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=['Low', 'Medium', 'High', 'Premium'],
    include_lowest=True
)

# ── Doctor load proxy: patients per hospital (relative)
hospital_volume = tdf.groupby('Hospital')['Patient_ID'].transform('count')
tdf['Hospital_Patient_Volume'] = hospital_volume

print('Quality Tier distribution:')
print(tdf['Quality_Tier'].value_counts().to_string())

Quality Tier distribution:
Quality_Tier
Medium     9696
High       7079
Premium    4033
Low        3192


## 6 — Equity & Disparity Metrics

In [7]:
# ── Insurance-adjusted cost burden
# Uninsured patients paying above-average → financial distress signal
tdf['Financial_Distress_Flag'] = (
    (tdf['Insurance'] == 'No') &
    (tdf['Treatment_Cost'] > overall_avg_cost)
).astype(np.int8)

# ── Care equity gap: difference between uninsured and insured avg cost per diagnosis
insured_diag_avg   = tdf[tdf['Insured_Flag'] == 1].groupby('Diagnosis')['Treatment_Cost'].mean()
uninsured_diag_avg = tdf[tdf['Insured_Flag'] == 0].groupby('Diagnosis')['Treatment_Cost'].mean()

equity_gap = (insured_diag_avg - uninsured_diag_avg).rename('Insurance_Cost_Gap_by_Diagnosis')
print('Insurance cost gap by diagnosis (Insured − Uninsured mean):')
print(equity_gap.round(2).to_string())

# ── Mortality equity gap: mortality rate difference by economic status
mort_by_econ = (
    tdf.groupby('Economic_Status')['Deceased_Flag']
    .mean()
    .mul(100).round(2)
    .rename('Mortality_Rate_%')
)
print('\nMortality rate by Economic Status:')
print(mort_by_econ.to_string())

Insurance cost gap by diagnosis (Insured − Uninsured mean):
Diagnosis
Asthma          394.69
Covid           158.70
Diabetes        286.57
Flu            -335.62
Hypertension    306.34

Mortality rate by Economic Status:
Economic_Status
High     5.28
Low      5.27
Middle   4.28


## 7 — Preview the Enriched Fact Table

In [8]:
print(f'Enriched shape: {tdf.shape[0]:,} rows  ×  {tdf.shape[1]} columns')
print('\nNew columns added:')
new_cols = [c for c in tdf.columns if c not in df.columns]
for c in new_cols:
    print(f'  + {c}')

display(tdf[new_cols].describe(include='all').T)

Enriched shape: 24,000 rows  ×  36 columns

New columns added:
  + Recovered_Flag
  + Deceased_Flag
  + Under_Treatment_Flag
  + Insured_Flag
  + Private_Hospital_Flag
  + Male_Flag
  + Female_Flag
  + Vulnerable_Patient_Flag
  + Cost_vs_Overall_Avg
  + Cost_Pct_vs_Overall_Avg
  + Cost_vs_Diagnosis_Avg
  + Cost_vs_Hospital_Avg
  + Cost_Percentile_Rank
  + High_Cost_Flag
  + Operational_Access_Index
  + Quality_Tier
  + Hospital_Patient_Volume
  + Financial_Distress_Flag


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Recovered_Flag,"24,000.00",NaN,NaN,NaN,0.83,0.38,0.00,1.00,1.00,1.00,1.00
Deceased_Flag,"24,000.00",NaN,NaN,NaN,0.05,0.22,0.00,0.00,0.00,0.00,1.00
Under_Treatment_Flag,"24,000.00",NaN,NaN,NaN,0.13,0.33,0.00,0.00,0.00,0.00,1.00
Insured_Flag,"24,000.00",NaN,NaN,NaN,0.33,0.47,0.00,0.00,0.00,1.00,1.00
Private_Hospital_Flag,"24,000.00",NaN,NaN,NaN,0.67,0.47,0.00,0.00,1.00,1.00,1.00
Male_Flag,"24,000.00",NaN,NaN,NaN,0.50,0.50,0.00,0.00,0.00,1.00,1.00
Female_Flag,"24,000.00",NaN,NaN,NaN,0.50,0.50,0.00,0.00,1.00,1.00,1.00
Vulnerable_Patient_Flag,"24,000.00",NaN,NaN,NaN,0.09,0.29,0.00,0.00,0.00,0.00,1.00
Cost_vs_Overall_Avg,"24,000.00",NaN,NaN,NaN,0.00,"11,855.94","-23,768.92","-9,165.34","-1,435.61","7,872.56","48,076.59"
Cost_Pct_vs_Overall_Avg,"24,000.00",NaN,NaN,NaN,0.00,47.71,-95.65,-36.88,-5.78,31.68,193.47


## 8 — Executive KPI Scorecard

In [9]:
scorecard = pd.DataFrame([{
    # Volume
    'Total_Patients'            : len(tdf),
    'Unique_Hospitals'          : tdf['Hospital'].nunique(),
    'Unique_Diagnoses'          : tdf['Diagnosis'].nunique(),
    'Unique_Regions'            : tdf['Region'].nunique(),

    # Outcomes (%)
    'Recovery_Rate_Pct'         : round(tdf['Recovered_Flag'].mean() * 100, 2),
    'Mortality_Rate_Pct'        : round(tdf['Deceased_Flag'].mean() * 100, 2),
    'Under_Treatment_Rate_Pct'  : round(tdf['Under_Treatment_Flag'].mean() * 100, 2),

    # Cost
    'Avg_Treatment_Cost'        : round(overall_avg_cost, 2),
    'Median_Treatment_Cost'     : round(overall_median_cost, 2),
    'Cost_Std_Dev'              : round(tdf['Treatment_Cost'].std(), 2),
    'High_Cost_Patient_Pct'     : round(tdf['High_Cost_Flag'].mean() * 100, 2),

    # Payer Mix
    'Insurance_Coverage_Pct'    : round(tdf['Insured_Flag'].mean() * 100, 2),
    'Private_Hospital_Pct'      : round(tdf['Private_Hospital_Flag'].mean() * 100, 2),

    # Equity
    'Financial_Distress_Pct'    : round(tdf['Financial_Distress_Flag'].mean() * 100, 2),
    'Vulnerable_Patient_Pct'    : round(tdf['Vulnerable_Patient_Flag'].mean() * 100, 2),

    # Operations
    'Avg_Doctor_Availability'   : round(tdf['Doctor_Availability'].mean(), 2),
    'Avg_Cleanliness_Score'     : round(tdf['Cleanliness_Score'].mean(), 2),
    'Avg_Doctor_Experience_Yrs' : round(tdf['Doctor_Experience_Years'].mean(), 2),
    'Avg_Operational_Access_Idx': round(tdf['Operational_Access_Index'].mean(), 4),
}])

scorecard.to_csv(ANALYSIS_DIR / 'kpi_scorecard.csv', index=False)
print('✅ Saved kpi_scorecard.csv')
display(scorecard.T.rename(columns={0: 'Value'}))

✅ Saved kpi_scorecard.csv


,Value
Total_Patients,"24,000.00"
Unique_Hospitals,3.00
Unique_Diagnoses,5.00
Unique_Regions,4.00
Recovery_Rate_Pct,82.56
Mortality_Rate_Pct,4.88
Under_Treatment_Rate_Pct,12.56
Avg_Treatment_Cost,"24,849.47"
Median_Treatment_Cost,"23,413.86"
Cost_Std_Dev,"11,855.94"


## 9 — Dimensional Summary Tables

In [10]:
def build_summary(data: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Aggregate KPIs by the given grouping columns."""
    out = data.groupby(group_cols, dropna=False, observed=True).agg(
        Patient_Volume          = ('Patient_ID',               'count'),
        Avg_Age                 = ('Age',                      'mean'),

        # -- Cost KPIs --
        Avg_Treatment_Cost      = ('Treatment_Cost',           'mean'),
        Median_Treatment_Cost   = ('Treatment_Cost',           'median'),
        Std_Treatment_Cost      = ('Treatment_Cost',           'std'),
        Min_Treatment_Cost      = ('Treatment_Cost',           'min'),
        Max_Treatment_Cost      = ('Treatment_Cost',           'max'),
        High_Cost_Patient_Rate  = ('High_Cost_Flag',           'mean'),

        # -- Outcome KPIs --
        Recovery_Rate           = ('Recovered_Flag',           'mean'),
        Mortality_Rate          = ('Deceased_Flag',            'mean'),
        Under_Treatment_Rate    = ('Under_Treatment_Flag',     'mean'),

        # -- Payer Mix --
        Insurance_Coverage_Rate = ('Insured_Flag',             'mean'),
        Private_Hospital_Rate   = ('Private_Hospital_Flag',    'mean'),

        # -- Equity --
        Financial_Distress_Rate = ('Financial_Distress_Flag',  'mean'),
        Vulnerable_Patient_Rate = ('Vulnerable_Patient_Flag',  'mean'),

        # -- Operations --
        Avg_Doctor_Availability = ('Doctor_Availability',      'mean'),
        Avg_Cleanliness_Score   = ('Cleanliness_Score',        'mean'),
        Avg_Doctor_Experience   = ('Doctor_Experience_Years',  'mean'),
        Avg_Operational_Index   = ('Operational_Access_Index', 'mean'),
    ).reset_index()

    # Convert rates (0–1) → percentages
    rate_cols = [
        'Recovery_Rate', 'Mortality_Rate', 'Under_Treatment_Rate',
        'Insurance_Coverage_Rate', 'Private_Hospital_Rate',
        'Financial_Distress_Rate', 'Vulnerable_Patient_Rate',
        'High_Cost_Patient_Rate',
    ]
    out[rate_cols] = (out[rate_cols] * 100).round(2)

    # Round all remaining numeric columns
    for col in out.select_dtypes(include='number').columns:
        if col not in rate_cols:
            out[col] = out[col].round(2)

    # Add volume share
    out['Volume_Share_Pct'] = (out['Patient_Volume'] / out['Patient_Volume'].sum() * 100).round(2)

    return out


summary_specs = {
    'tableau_hospital_summary.csv'         : ['Hospital', 'Hospital_Type'],
    'tableau_diagnosis_summary.csv'        : ['Diagnosis'],
    'tableau_region_summary.csv'           : ['Region'],
    'tableau_age_group_summary.csv'        : ['Age_Group'],
    'tableau_economic_status_summary.csv'  : ['Economic_Status'],
    'tableau_insurance_summary.csv'        : ['Insurance'],
    'tableau_cost_tier_summary.csv'        : ['Cost_Tier'],
}

for filename, group_cols in summary_specs.items():
    table = build_summary(tdf, group_cols)
    table.to_csv(ANALYSIS_DIR / filename, index=False)
    print(f'\n📂 {filename}  ({len(table)} rows)')
    display(table)


📂 tableau_hospital_summary.csv  (3 rows)


,Hospital,Hospital_Type,Patient_Volume,Avg_Age,Avg_Treatment_Cost,Median_Treatment_Cost,Std_Treatment_Cost,Min_Treatment_Cost,Max_Treatment_Cost,High_Cost_Patient_Rate,Recovery_Rate,Mortality_Rate,Under_Treatment_Rate,Insurance_Coverage_Rate,Private_Hospital_Rate,Financial_Distress_Rate,Vulnerable_Patient_Rate,Avg_Doctor_Availability,Avg_Cleanliness_Score,Avg_Doctor_Experience,Avg_Operational_Index,Volume_Share_Pct
0,Carepoint,Private,8000,51.10,"24,118.61","22,633.64","11,130.20","1,188.60","61,706.67",7.22,82.56,4.88,12.56,33.29,100.00,28.56,9.25,5.64,2.78,16.97,0.49,33.33
1,Medilife,Government,8000,51.10,"21,926.01","20,576.04","10,118.37","1,080.55","56,096.97",4.01,82.56,4.88,12.56,33.29,0.00,24.00,9.25,5.64,2.78,19.90,0.51,33.33
2,Sunrise,Private,8000,51.10,"28,503.81","26,748.84","13,153.88","1,404.72","72,926.06",18.76,82.56,4.88,12.56,33.29,100.00,37.94,9.25,5.64,2.78,19.90,0.51,33.33



📂 tableau_diagnosis_summary.csv  (5 rows)


,Diagnosis,Patient_Volume,Avg_Age,Avg_Treatment_Cost,Median_Treatment_Cost,Std_Treatment_Cost,Min_Treatment_Cost,Max_Treatment_Cost,High_Cost_Patient_Rate,Recovery_Rate,Mortality_Rate,Under_Treatment_Rate,Insurance_Coverage_Rate,Private_Hospital_Rate,Financial_Distress_Rate,Vulnerable_Patient_Rate,Avg_Doctor_Availability,Avg_Cleanliness_Score,Avg_Doctor_Experience,Avg_Operational_Index,Volume_Share_Pct
0,Asthma,3555,50.71,"20,104.58","19,329.07","5,198.92","6,459.02","43,370.95",0.03,83.29,3.29,13.42,33.42,66.67,10.18,9.11,5.71,2.85,18.59,0.51,14.81
1,Covid,4899,51.67,"40,841.61","39,270.97","8,504.29","20,576.42","72,926.06",42.99,79.49,11.94,8.57,33.01,66.67,66.36,9.74,5.64,2.76,19.12,0.50,20.41
2,Diabetes,5016,50.74,"29,290.38","28,222.82","6,663.40","12,999.09","54,621.81",5.30,84.09,2.93,12.98,33.19,66.67,50.00,8.97,5.65,2.83,19.01,0.50,20.90
3,Flu,5817,51.03,"11,631.69","11,114.12","4,036.36","1,080.55","29,997.94",0.00,82.31,3.25,14.44,32.75,66.67,0.29,8.92,5.62,2.75,18.87,0.50,24.24
4,Hypertension,4713,51.26,"23,392.84","22,528.27","5,594.82","9,291.35","47,038.36",0.57,83.90,2.80,13.30,34.25,66.67,23.38,9.55,5.58,2.73,18.94,0.50,19.64



📂 tableau_region_summary.csv  (4 rows)


...

,Economic_Status,Patient_Volume,Avg_Age,Avg_Treatment_Cost,Median_Treatment_Cost,Std_Treatment_Cost,Min_Treatment_Cost,Max_Treatment_Cost,High_Cost_Patient_Rate,Recovery_Rate,Mortality_Rate,Under_Treatment_Rate,Insurance_Coverage_Rate,Private_Hospital_Rate,Financial_Distress_Rate,Vulnerable_Patient_Rate,Avg_Doctor_Availability,Avg_Cleanliness_Score,Avg_Doctor_Experience,Avg_Operational_Index,Volume_Share_Pct
0,High,4887,51.54,"26,633.76","25,337.97","12,559.82","2,312.63","72,926.06",13.81,82.20,5.28,12.52,33.39,66.67,33.54,0.00,5.45,2.73,19.05,0.49,20.36
1,Low,9444,51.29,"22,100.16","20,968.94","10,443.02","1,080.55","59,470.00",4.97,81.61,5.27,13.12,32.56,66.67,24.84,23.51,5.78,2.80,18.95,0.51,39.35
2,Middle,9669,50.69,"26,632.98","25,327.47","12,268.10","2,263.04","72,277.11",12.99,83.68,4.28,12.04,33.94,66.67,33.66,0.00,5.58,2.78,18.84,0.50,40.29



📂 tableau_insurance_summary.csv  (2 rows)


,Insurance,Patient_Volume,Avg_Age,Avg_Treatment_Cost,Median_Treatment_Cost,Std_Treatment_Cost,Min_Treatment_Cost,Max_Treatment_Cost,High_Cost_Patient_Rate,Recovery_Rate,Mortality_Rate,Under_Treatment_Rate,Insurance_Coverage_Rate,Private_Hospital_Rate,Financial_Distress_Rate,Vulnerable_Patient_Rate,Avg_Doctor_Availability,Avg_Cleanliness_Score,Avg_Doctor_Experience,Avg_Operational_Index,Volume_Share_Pct
0,No,16011,51.33,"24,800.53","23,270.28","11,836.63","1,882.50","72,926.06",9.92,82.74,4.80,12.46,0.00,66.67,45.22,13.87,5.61,2.77,19.05,0.50,66.71
1,Yes,7989,50.63,"24,947.56","23,664.63","11,894.67","1,080.55","70,013.68",10.16,82.20,5.03,12.77,100.00,66.67,0.00,0.00,5.68,2.79,18.66,0.50,33.29



📂 tableau_cost_tier_summary.csv  (4 rows)


,Cost_Tier,Patient_Volume,Avg_Age,Avg_Treatment_Cost,Median_Treatment_Cost,Std_Treatment_Cost,Min_Treatment_Cost,Max_Treatment_Cost,High_Cost_Patient_Rate,Recovery_Rate,Mortality_Rate,Under_Treatment_Rate,Insurance_Coverage_Rate,Private_Hospital_Rate,Financial_Distress_Rate,Vulnerable_Patient_Rate,Avg_Doctor_Availability,Avg_Cleanliness_Score,Avg_Doctor_Experience,Avg_Operational_Index,Volume_Share_Pct
0,High,6000,52.33,"27,498.09","27,262.44","3,857.13","20,576.42","36,860.97",0.00,82.10,4.60,13.30,33.70,66.67,48.55,9.60,5.74,2.80,19.10,0.51,25.00
1,Low,6000,47.35,"11,105.42","11,114.12","3,151.09","1,080.55","18,237.53",0.00,82.40,2.90,14.70,33.20,66.67,0.00,8.95,5.09,2.61,18.77,0.47,25.00
2,Medium,6000,50.21,"20,109.82","19,868.08","3,096.76","14,028.97","26,748.34",0.00,84.00,3.25,12.75,31.70,66.67,6.67,10.45,5.05,2.60,18.66,0.47,25.00
3,Very High,6000,54.50,"40,684.56","38,970.42","7,662.90","28,355.70","72,926.06",40.00,81.75,8.75,9.50,34.55,66.67,65.45,8.00,6.66,3.10,19.16,0.56,25.00


## 10 — Cross-Dimensional Analysis (Diagnosis × Age Group × Insurance)

In [11]:
cross_dim = build_summary(tdf, ['Diagnosis', 'Age_Group', 'Insurance'])
cross_dim.to_csv(ANALYSIS_DIR / 'tableau_cross_dim_summary.csv', index=False)
print(f'✅ Saved tableau_cross_dim_summary.csv  ({len(cross_dim)} rows)')
display(cross_dim)

✅ Saved tableau_cross_dim_summary.csv  (30 rows)


,Diagnosis,Age_Group,Insurance,Patient_Volume,Avg_Age,Avg_Treatment_Cost,Median_Treatment_Cost,Std_Treatment_Cost,Min_Treatment_Cost,Max_Treatment_Cost,High_Cost_Patient_Rate,Recovery_Rate,Mortality_Rate,Under_Treatment_Rate,Insurance_Coverage_Rate,Private_Hospital_Rate,Financial_Distress_Rate,Vulnerable_Patient_Rate,Avg_Doctor_Availability,Avg_Cleanliness_Score,Avg_Doctor_Experience,Avg_Operational_Index,Volume_Share_Pct
0,Asthma,Middle-Aged,No,1005,48.05,"20,013.03","19,287.77","4,853.04","8,426.43","37,989.08",0.00,83.88,1.19,14.93,0.00,66.67,15.02,0.00,5.63,2.88,18.26,0.51,4.19
1,Asthma,Middle-Aged,Yes,528,47.89,"20,007.59","19,319.68","5,041.65","8,367.14","40,973.62",0.00,86.36,0.00,13.64,100.00,66.67,0.00,0.00,5.84,2.78,17.94,0.50,2.20
2,Asthma,Senior,No,768,71.78,"21,816.37","21,487.49","5,168.47","10,594.26","43,370.95",0.13,80.08,9.38,10.55,0.00,66.67,23.83,42.19,5.68,2.81,19.17,0.50,3.20
3,Asthma,Senior,Yes,393,72.66,"22,594.59","22,125.58","5,450.14","10,963.65","40,647.48",0.00,80.15,8.40,11.45,100.00,66.67,0.00,0.00,5.82,2.70,17.95,0.49,1.64
4,Asthma,Young Adult,No,594,26.42,"17,520.66","17,522.44","4,287.95","6,459.02","30,837.81",0.00,85.35,0.00,14.65,0.00,66.67,4.71,0.00,5.86,2.97,19.01,0.52,2.48
5,Asthma,Young Adult,Yes,267,27.35,"17,800.57","17,522.44","4,709.11","6,794.19","35,583.33",0.00,84.27,0.00,15.73,100.00,66.67,0.00,0.00,5.34,2.93,19.44,0.51,1.11
6,Covid,Middle-Aged,No,1398,48.45,"40,782.15","39,031.90","8,470.53","21,864.07","71,522.56",43.13,79.61,11.37,9.01,0.00,66.67,99.36,0.00,5.79,2.81,19.09,0.51,5.82
7,Covid,Middle-Aged,Yes,639,49.10,"40,839.79","38,972.05","8,553.65","21,917.19","70,013.68",42.88,75.59,14.55,9.86,100.00,66.67,0.00,0.00,5.41,2.69,19.82,0.49,2.66
8,Covid,Senior,No,1137,73.09,"42,591.08","41,112.42","8,759.33","23,876.37","72,926.06",49.16,79.68,13.46,6.86,0.00,66.67,99.74,41.95,5.60,2.66,19.99,0.50,4.74
9,Covid,Senior,Yes,537,72.56,"42,846.69","41,745.24","8,245.41","24,001.21","69,418.31",51.77,74.30,18.99,6.70,100.00,66.67,0.00,0.00,5.61,2.71,19.47,0.50,2.24


## 11 — Top/Bottom Performers

In [12]:
hosp_summary = build_summary(tdf, ['Hospital', 'Hospital_Type'])

print('━━ Top 5 Hospitals by Recovery Rate ━━')
display(hosp_summary.nlargest(5, 'Recovery_Rate')[['Hospital', 'Hospital_Type',
    'Patient_Volume', 'Recovery_Rate', 'Mortality_Rate',
    'Avg_Treatment_Cost', 'Avg_Operational_Index']])

print('\n━━ Bottom 5 Hospitals by Recovery Rate ━━')
display(hosp_summary.nsmallest(5, 'Recovery_Rate')[['Hospital', 'Hospital_Type',
    'Patient_Volume', 'Recovery_Rate', 'Mortality_Rate',
    'Avg_Treatment_Cost', 'Avg_Operational_Index']])

print('\n━━ Top 5 Hospitals by Mortality Rate ━━')
display(hosp_summary.nlargest(5, 'Mortality_Rate')[['Hospital', 'Hospital_Type',
    'Patient_Volume', 'Mortality_Rate', 'Financial_Distress_Rate',
    'Avg_Treatment_Cost', 'Avg_Operational_Index']])

━━ Top 5 Hospitals by Recovery Rate ━━


,Hospital,Hospital_Type,Patient_Volume,Recovery_Rate,Mortality_Rate,Avg_Treatment_Cost,Avg_Operational_Index
0,Carepoint,Private,8000,82.56,4.88,"24,118.61",0.49
1,Medilife,Government,8000,82.56,4.88,"21,926.01",0.51
2,Sunrise,Private,8000,82.56,4.88,"28,503.81",0.51



━━ Bottom 5 Hospitals by Recovery Rate ━━


,Hospital,Hospital_Type,Patient_Volume,Recovery_Rate,Mortality_Rate,Avg_Treatment_Cost,Avg_Operational_Index
0,Carepoint,Private,8000,82.56,4.88,"24,118.61",0.49
1,Medilife,Government,8000,82.56,4.88,"21,926.01",0.51
2,Sunrise,Private,8000,82.56,4.88,"28,503.81",0.51



━━ Top 5 Hospitals by Mortality Rate ━━


,Hospital,Hospital_Type,Patient_Volume,Mortality_Rate,Financial_Distress_Rate,Avg_Treatment_Cost,Avg_Operational_Index
0,Carepoint,Private,8000,4.88,28.56,"24,118.61",0.49
1,Medilife,Government,8000,4.88,24.00,"21,926.01",0.51
2,Sunrise,Private,8000,4.88,37.94,"28,503.81",0.51


## 12 — Diagnosis Risk Profile

In [13]:
diag_risk = (
    tdf.groupby('Diagnosis', observed=True)
    .agg(
        Patient_Volume      = ('Patient_ID', 'count'),
        Mortality_Rate_Pct  = ('Deceased_Flag', lambda x: round(x.mean()*100, 2)),
        Recovery_Rate_Pct   = ('Recovered_Flag', lambda x: round(x.mean()*100, 2)),
        Avg_Cost            = ('Treatment_Cost', 'mean'),
        Cost_Std            = ('Treatment_Cost', 'std'),
        Uninsured_Pct       = ('Insured_Flag', lambda x: round((1 - x.mean())*100, 2)),
        Distress_Pct        = ('Financial_Distress_Flag', lambda x: round(x.mean()*100, 2)),
    )
    .round(2)
    .reset_index()
    .sort_values('Mortality_Rate_Pct', ascending=False)
)

print('Diagnosis risk profile (sorted by mortality rate):')
display(diag_risk)

Diagnosis risk profile (sorted by mortality rate):


,Diagnosis,Patient_Volume,Mortality_Rate_Pct,Recovery_Rate_Pct,Avg_Cost,Cost_Std,Uninsured_Pct,Distress_Pct
1,Covid,4899,11.94,79.49,"40,841.61","8,504.29",66.99,66.36
0,Asthma,3555,3.29,83.29,"20,104.58","5,198.92",66.58,10.18
3,Flu,5817,3.25,82.31,"11,631.69","4,036.36",67.25,0.29
2,Diabetes,5016,2.93,84.09,"29,290.38","6,663.40",66.81,50.00
4,Hypertension,4713,2.80,83.90,"23,392.84","5,594.82",65.75,23.38


## 13 — Save Enriched Fact Table

In [14]:
# Drop internal lineage column before writing the fact table
fact_table = tdf.drop(columns=['_source_file'], errors='ignore')

output_path = PROCESSED_DIR / 'hospital_tableau_ready.csv'
fact_table.to_csv(output_path, index=False)

print(f'✅ Saved: {output_path}')
print(f'   Rows      : {len(fact_table):,}')
print(f'   Columns   : {fact_table.shape[1]}')
print(f'   File size : {output_path.stat().st_size / 1_048_576:.2f} MB')

✅ Saved: /Users/harshita/SectionB_G11_HealthOps_Analytics/data/processed/hospital_tableau_ready.csv
   Rows      : 24,000
   Columns   : 35
   File size : 4.16 MB


## 14 — Schema / Data Dictionary

In [15]:
COLUMN_DOCS = {
    # Original
    'Patient_ID'               : 'Unique patient identifier',
    'Age'                      : 'Patient age in years',
    'Gender'                   : 'Patient gender (Male / Female)',
    'Hospital'                 : 'Hospital name',
    'Hospital_ID'              : 'Unique hospital identifier',
    'Hospital_Type'            : 'Government or Private',
    'Diagnosis'                : 'Primary diagnosis category',
    'Treatment_Cost'           : 'Total treatment cost in ₹ INR',
    'Doctor_Experience_Years'  : 'Treating doctors years of experience',
    'Doctor_Availability'      : 'Availability score (higher = more available)',
    'Cleanliness_Score'        : 'Facility cleanliness rating',
    'Economic_Status'          : 'Patient economic status (Low / Medium / High)',
    'Outcome'                  : 'Treatment outcome (Recovered / Deceased / Under Treatment)',
    'Insurance'                : 'Whether patient is insured (Yes / No)',
    'Region'                   : 'Geographic region (North / South / East / West)',
    'Age_Group'                : 'Derived age band from cleaning step',
    'Cost_Tier'                : 'Derived cost band from cleaning step',
    # Engineered flags
    'Recovered_Flag'           : '1 if Outcome == Recovered',
    'Deceased_Flag'            : '1 if Outcome == Deceased',
    'Under_Treatment_Flag'     : '1 if Outcome == Under Treatment',
    'Insured_Flag'             : '1 if Insurance == Yes',
    'Private_Hospital_Flag'    : '1 if Hospital_Type == Private',
    'Male_Flag'                : '1 if Gender == Male',
    'Female_Flag'              : '1 if Gender == Female',
    'Vulnerable_Patient_Flag'  : '1 if Senior + Low income + Uninsured',
    'Financial_Distress_Flag'  : '1 if Uninsured AND cost > overall mean',
    'High_Cost_Flag'           : '1 if Treatment_Cost ≥ 90th percentile',
    # Cost KPIs
    'Cost_vs_Overall_Avg'      : 'Treatment_Cost minus overall mean (₹)',
    'Cost_Pct_vs_Overall_Avg'  : '% deviation from overall mean cost',
    'Cost_vs_Diagnosis_Avg'    : 'Treatment_Cost minus within-diagnosis mean (₹)',
    'Cost_vs_Hospital_Avg'     : 'Treatment_Cost minus within-hospital mean (₹)',
    'Cost_Percentile_Rank'     : 'Patients cost percentile (0–100) globally',
    # Operational
    'Operational_Access_Index' : 'Weighted composite: 50% avail + 30% clean + 20% exp (0–1)',
    'Quality_Tier'             : 'Quartile band of Operational_Access_Index (Low/Medium/High/Premium)',
    'Hospital_Patient_Volume'  : 'Total patients seen at same hospital in this dataset',
}

schema = pd.DataFrame([
    {
        'Column'     : col,
        'Dtype'      : str(fact_table[col].dtype) if col in fact_table.columns else 'N/A',
        'Null_Count' : int(fact_table[col].isnull().sum()) if col in fact_table.columns else 'N/A',
        'Unique_Values': int(fact_table[col].nunique()) if col in fact_table.columns else 'N/A',
        'Description': COLUMN_DOCS.get(col, ''),
    }
    for col in fact_table.columns
])

schema.to_csv(ANALYSIS_DIR / 'schema_manifest.csv', index=False)
print('✅ Saved schema_manifest.csv')
display(schema)

✅ Saved schema_manifest.csv


,Column,Dtype,Null_Count,Unique_Values,Description
0,Patient_ID,str,0,8000,Unique patient identifier
1,Age,float64,0,67,Patient age in years
2,Gender,str,0,2,Patient gender (Male / Female)
3,Hospital,str,0,3,Hospital name
4,Hospital_Type,str,0,2,Government or Private
5,Diagnosis,str,0,5,Primary diagnosis category
6,Treatment_Cost,float64,0,20919,Total treatment cost in ₹ INR
7,Doctor_Experience_Years,float64,0,40,Treating doctors years of experience
8,Doctor_Availability,int64,0,14,Availability score (higher = more available)
9,Cleanliness_Score,int64,0,5,Facility cleanliness rating


## 15 — Output Manifest

In [16]:
all_outputs = [
    PROCESSED_DIR / 'hospital_tableau_ready.csv',
    *sorted(ANALYSIS_DIR.glob('*.csv'))
]

# Use simple print without complex f-string nesting to avoid syntax issues across python versions
print('{:<50} {:>10}  {:>8}'.format('File', 'Size', 'Rows'))
print('─' * 72)
for p in all_outputs:
    if p.exists():
        try:
            rows = sum(1 for _ in open(p)) - 1   # subtract header
        except Exception:
            rows = '?'
        size_kb = p.stat().st_size / 1024
        print('{:<50} {:>8.1f} KB  {:>8}'.format(p.name, size_kb, rows))
    else:
        print('{:<50} ⚠️  NOT FOUND'.format(p.name))
print('─' * 72)


File                                                     Size      Rows
────────────────────────────────────────────────────────────────────────
hospital_tableau_ready.csv                           4258.0 KB     24000
age_group_kpis.csv                                      0.4 KB         3
diagnosis_kpis.csv                                      0.5 KB         5
economic_status_kpis.csv                                0.4 KB         3
hospital_kpis.csv                                       0.4 KB         3
insight_candidates.csv                                  0.9 KB         5
insurance_kpis.csv                                      0.3 KB         2
kpi_scorecard.csv                                       0.5 KB         1
outcome_mix_by_diagnosis.csv                            0.2 KB         5
outcome_mix_by_hospital.csv                             0.1 KB         3
pairwise_diagnosis_cost_tests.csv                       0.6 KB        10
region_kpis.csv                                     

---
## Tableau Handoff Notes

### Recommended Dashboard Views

| # | Dashboard | Primary Fields | Suggested Chart Types |
|---|---|---|---|
| 1 | **Executive Scorecard** | Recovery_Rate, Mortality_Rate, Avg_Treatment_Cost, Insurance_Coverage_Rate | KPI tiles, sparklines |
| 2 | **Cost Drivers** | Cost_vs_Diagnosis_Avg, Cost_Tier, Hospital_Type, Age_Group | Bar chart, heat-map, scatter |
| 3 | **Outcome Risk** | Outcome mix, Diagnosis, Age_Group, Economic_Status | Stacked bar, tree-map |
| 4 | **Operations Quality** | Operational_Access_Index, Quality_Tier, Doctor_Availability | Gauge, bullet chart |
| 5 | **Equity / Disparity** | Financial_Distress_Flag, Vulnerable_Patient_Flag, Mortality_Rate × Economic_Status | Side-by-side bar |
| 6 | **Hospital Benchmarking** | Hospital_Patient_Volume, Recovery_Rate, High_Cost_Patient_Rate | Scatter plot, rank table |

### Recommended Filters / Parameters
- Hospital, Hospital_Type, Region, Diagnosis, Age_Group, Gender
- Economic_Status, Insurance, Cost_Tier, Quality_Tier, Outcome

### Key Calculated Fields for Tableau
- `Net Recovery Efficiency` = `Recovery_Rate / Avg_Treatment_Cost × 1000`
- `Cost-to-Outcome Score` = `Avg_Treatment_Cost × Mortality_Rate`
- `Equity Index` = `1 − Financial_Distress_Rate`
